# Multi-seed Results Inspection

**Research question:** Does representation stabilization imply representation sufficiency?

This notebook covers all 15 runs (3 learning rates × 5 seeds) plus two new analyses:
- **Cross-learning-rate CKA**: tests whether different LRs converge to similar or different representation geometries.
- **Spectral component analysis**: tests whether early probe performance is concentrated in dominant singular directions.

Run all cells top-to-bottom. Final cell exports `notebooks/.generated/dashboard.html`.

| Section | Contents |
|---------|----------|
| 1 | Run inventory |
| 2 | Final performance across seeds |
| 3 | Primary evidence: probes vs CKA |
| 4 | Learning-rate comparison |
| 5 | Within-LR seed robustness |
| 6 | Cross-learning-rate CKA |
| 7 | Spectral component analysis |
| 8 | Neural Collapse |
| 9 | Last-layer eNTK |
| 10 | Operational timing summaries |

In [ ]:
import os, base64, io, warnings
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import HTML, display
warnings.filterwarnings('ignore')

matplotlib.rcParams['figure.dpi'] = 110
matplotlib.rcParams['axes.grid'] = True
matplotlib.rcParams['grid.alpha'] = 0.3
matplotlib.rcParams['axes.spines.top'] = False
matplotlib.rcParams['axes.spines.right'] = False

try:
    NB_DIR = Path(os.path.dirname(os.path.abspath(__file__)))
except NameError:
    NB_DIR = Path.cwd()

REPO_ROOT   = NB_DIR.parent
RESULTS_DIR = REPO_ROOT / 'thesis_results'
AGG_DIR     = RESULTS_DIR / '_aggregate'
GEN_DIR     = NB_DIR / '.generated'
SPECTRAL_PNG_DIR = AGG_DIR / 'plots' / 'spectral_png'

GEN_DIR.mkdir(exist_ok=True)
SPECTRAL_PNG_DIR.mkdir(parents=True, exist_ok=True)

LR_LIST    = ['0.05', '0.10', '0.20']
SEED_LIST  = [0, 1, 2, 3, 42]
LR_COLORS  = {'0.05': '#1f77b4', '0.10': '#ff7f0e', '0.20': '#2ca02c'}
LR_TAG     = {'0.05': 'lr005', '0.10': 'lr010', '0.20': 'lr020'}

def run_name(lr, seed):
    return f'final_{LR_TAG[lr]}_seed{seed}'

print(f'RESULTS_DIR = {RESULTS_DIR}')
print(f'AGG_DIR     = {AGG_DIR}')
print(f'AGG_DIR exists: {AGG_DIR.exists()}')

In [ ]:
# ── Run inventory ─────────────────────────────────────────────────────────────
ALL_RUNS = {}
for lr in LR_LIST:
    for seed in SEED_LIST:
        rname = run_name(lr, seed)
        rpath = RESULTS_DIR / rname
        ALL_RUNS[(lr, seed)] = {
            'name': rname, 'path': rpath,
            'exists': rpath.exists(), 'lr': lr, 'seed': seed,
        }

print(f"{'Run':<32} {'Status':>7}")
print('-' * 41)
for (lr, seed), info in ALL_RUNS.items():
    print(f"{info['name']:<32} {'OK' if info['exists'] else 'MISSING':>7}")

n_ok = sum(1 for v in ALL_RUNS.values() if v['exists'])
print(f'\n{n_ok}/15 run directories found.')

In [ ]:
# ── Load per-run CSVs ─────────────────────────────────────────────────────────
CSV_NAMES = [
    'master_trajectory.csv', 'cka_summary.csv',
    'neural_collapse.csv', 'entk_summary.csv',
    'probe_results_wide.csv', 'probe_results_long.csv',
]

dfs = {}  # dfs[(lr, seed)][csv_name] = DataFrame
for (lr, seed), info in ALL_RUNS.items():
    dfs[(lr, seed)] = {}
    if not info['exists']:
        continue
    res_dir = info['path'] / 'results'
    for csv_name in CSV_NAMES:
        fp = res_dir / csv_name
        if fp.exists():
            try:
                dfs[(lr, seed)][csv_name] = pd.read_csv(fp)
            except Exception as e:
                print(f'  WARN load {fp.name} ({lr},{seed}): {e}')

loaded = sum(len(d) for d in dfs.values())
print(f'Loaded {loaded} CSVs across {len(dfs)} runs.')
for lr in LR_LIST:
    n = sum(1 for s in SEED_LIST if 'master_trajectory.csv' in dfs.get((lr, s), {}))
    print(f'  LR={lr}: master_trajectory available for {n}/5 seeds')

In [ ]:
# ── Load aggregate CSVs ───────────────────────────────────────────────────────
agg = {}

def _load_agg(rel_path, key):
    fp = AGG_DIR / rel_path
    if fp.exists():
        try:
            agg[key] = pd.read_csv(fp)
            print(f'  OK   {key}: {agg[key].shape}')
        except Exception as e:
            agg[key] = None
            print(f'  WARN {key}: {e}')
    else:
        agg[key] = None
        print(f'  MISS {key}: {fp}')

print('=== Cross-LR CKA ===')
_load_agg('cross_lr_cka/cross_lr_cka_same_epoch_summary.csv', 'cross_lr_same_epoch')
_load_agg('cross_lr_cka/cross_lr_cka_final_summary.csv',      'cross_lr_final')
_load_agg('cross_lr_cka/combined_final_similarity_summary.csv','combined_final_sim')
_load_agg('cross_lr_cka/final_epoch_cka_matrix.csv',          'final_cka_matrix')
_load_agg('cross_lr_cka/within_lr_cross_seed_final_summary.csv','within_lr_final')
_load_agg('cross_lr_cka/cross_lr_cka_heatmap_summary.csv',    'cross_lr_heatmap_summary')

print('=== Spectral ===')
_load_agg('spectral_component_summary.csv', 'spectral_summary')
_load_agg('spectral_probe_results.csv',     'spectral_probes')
_load_agg('spectral_singular_values.csv',   'spectral_sv')

n_cross = sum(1 for k in ['cross_lr_same_epoch','cross_lr_final','combined_final_sim','final_cka_matrix','within_lr_final'] if agg.get(k) is not None)
n_spec  = sum(1 for k in ['spectral_summary','spectral_probes','spectral_sv'] if agg.get(k) is not None)
print(f'\nCross-LR CKA: {n_cross}/5 files   Spectral: {n_spec}/3 files')

In [ ]:
# ── Generate spectral PNG plots from CSV data ─────────────────────────────────
# Regenerates from data rather than converting PDFs; skips if PNG already exists.

def _lr_float(lr_str): return float(lr_str)

spectral_pngs = {}

# 1. Probe trajectories
out = SPECTRAL_PNG_DIR / 'probe_trajectories.png'
if not out.exists() and agg.get('spectral_probes') is not None:
    sp = agg['spectral_probes']
    sp = sp[sp['probe'] == 'logistic_regression'].copy()
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
    comp_style = [('full','#1a237e','-'), ('main','#2e7d32','--'), ('residual','#b71c1c',':')]
    for i, lr in enumerate(LR_LIST):
        ax = axes[i]
        sub = sp[sp['lr'] == _lr_float(lr)]
        for comp, color, ls in comp_style:
            cs = sub[sub['component'] == comp].groupby('epoch')['test_acc'].agg(['mean','std']).reset_index()
            if cs.empty: continue
            ax.plot(cs['epoch'], cs['mean'], color=color, ls=ls, lw=1.8, label=comp)
            ax.fill_between(cs['epoch'], cs['mean']-cs['std'], cs['mean']+cs['std'], alpha=0.15, color=color)
        ax.set_title(f'LR = {lr}', fontsize=11, fontweight='bold')
        ax.set_xlabel('Epoch', fontsize=9)
        if i == 0: ax.set_ylabel('Logistic probe test accuracy', fontsize=9)
        ax.legend(fontsize=8)
        ax.set_ylim(0, 1)
    fig.suptitle('Spectral Component Probe Trajectories (mean ± std across seeds)', fontsize=12, fontweight='bold')
    plt.tight_layout()
    fig.savefig(out, dpi=130, bbox_inches='tight')
    plt.close(fig)
spectral_pngs['probe_trajectories'] = out if out.exists() else None

# 2. k_main over time
out = SPECTRAL_PNG_DIR / 'k_main_over_time.png'
if not out.exists() and agg.get('spectral_summary') is not None:
    ss = agg['spectral_summary']
    fig, ax = plt.subplots(figsize=(8, 4))
    for lr in LR_LIST:
        sub = ss[ss['lr'] == _lr_float(lr)].groupby('epoch')['k_main'].agg(['mean','std']).reset_index()
        ax.plot(sub['epoch'], sub['mean'], color=LR_COLORS[lr], lw=1.8, label=f'LR={lr}')
        ax.fill_between(sub['epoch'], sub['mean']-sub['std'], sub['mean']+sub['std'], alpha=0.15, color=LR_COLORS[lr])
    ax.set_xlabel('Epoch', fontsize=10)
    ax.set_ylabel('k_main (dirs. for 80% singular mass)', fontsize=9)
    ax.set_title('Number of Dominant Singular Directions Over Training', fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)
    plt.tight_layout()
    fig.savefig(out, dpi=130, bbox_inches='tight')
    plt.close(fig)
spectral_pngs['k_main_over_time'] = out if out.exists() else None

# 3. Spectral mass over time
out = SPECTRAL_PNG_DIR / 'spectral_mass_over_time.png'
if not out.exists() and agg.get('spectral_summary') is not None:
    ss = agg['spectral_summary']
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, col, lbl in [(axes[0],'main_singular_mass','Main singular mass'), (axes[1],'main_energy_mass','Main energy mass')]:
        for lr in LR_LIST:
            sub = ss[ss['lr'] == _lr_float(lr)].groupby('epoch')[col].agg(['mean','std']).reset_index()
            ax.plot(sub['epoch'], sub['mean'], color=LR_COLORS[lr], lw=1.8, label=f'LR={lr}')
            ax.fill_between(sub['epoch'], sub['mean']-sub['std'], sub['mean']+sub['std'], alpha=0.15, color=LR_COLORS[lr])
        ax.axhline(0.8, color='gray', ls='--', lw=1, label='threshold=0.8')
        ax.set_xlabel('Epoch', fontsize=9); ax.set_ylabel(lbl, fontsize=9)
        ax.set_title(lbl, fontsize=10, fontweight='bold'); ax.legend(fontsize=8)
    fig.suptitle('Spectral Mass of Dominant Directions Over Training', fontsize=12, fontweight='bold')
    plt.tight_layout()
    fig.savefig(out, dpi=130, bbox_inches='tight')
    plt.close(fig)
spectral_pngs['spectral_mass_over_time'] = out if out.exists() else None

# 4. Singular spectrum snapshots (secondary/optional)
out = SPECTRAL_PNG_DIR / 'singular_spectrum_snapshots.png'
if not out.exists() and agg.get('spectral_sv') is not None:
    sv = agg['spectral_sv']
    avail = sorted(sv['epoch'].unique())
    snaps = [e for e in [0, 10, 50, 100, 200] if e in avail]
    if not snaps: snaps = avail[::max(1,len(avail)//5)][:5]
    fig, axes = plt.subplots(1, len(snaps), figsize=(4*len(snaps), 4), sharey=False)
    if len(snaps) == 1: axes = [axes]
    for ax, ep in zip(axes, snaps):
        sub = sv[(sv['epoch'] == ep) & (sv['lr'] == 0.05)]
        by_idx = sub.groupby('component_index')['normalized_singular_mass'].mean().reset_index()
        by_idx = by_idx.sort_values('component_index').head(50)
        ax.bar(by_idx['component_index'], by_idx['normalized_singular_mass'], color='#1f77b4', alpha=0.7)
        ax.set_title(f'Epoch {ep}', fontsize=10)
        ax.set_xlabel('Component index', fontsize=8)
        ax.set_ylabel('Norm. singular mass', fontsize=8)
    fig.suptitle('Singular Spectrum Snapshots (LR=0.05, mean across seeds, top 50)', fontsize=11, fontweight='bold')
    plt.tight_layout()
    fig.savefig(out, dpi=130, bbox_inches='tight')
    plt.close(fig)
spectral_pngs['singular_spectrum_snapshots'] = out if out.exists() else None

for k, v in spectral_pngs.items():
    print(f'  {k}: {"OK" if v and v.exists() else "MISSING"}')

In [ ]:
# ── Shared helpers ────────────────────────────────────────────────────────────

def _get_traj(lr, seed):
    return dfs.get((lr, seed), {}).get('master_trajectory.csv')

def _agg_over_seeds(lr, col, csv='master_trajectory.csv'):
    """Return DataFrame with columns [epoch, mean, std, n] aggregated over seeds for given LR."""
    frames = []
    for seed in SEED_LIST:
        df = dfs.get((lr, seed), {}).get(csv)
        if df is not None and col in df.columns:
            frames.append(df[['epoch', col]].copy())
    if not frames:
        return pd.DataFrame(columns=['epoch','mean','std','n'])
    combined = pd.concat(frames)
    return combined.groupby('epoch')[col].agg(['mean','std','count']).reset_index().rename(columns={'count':'n'})

def _fig_to_b64(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight', dpi=110)
    plt.close(fig)
    buf.seek(0)
    return 'data:image/png;base64,' + base64.b64encode(buf.read()).decode()

def _img_b64(path):
    path = Path(path)
    if not path.exists():
        return None
    with open(path, 'rb') as f:
        return 'data:image/png;base64,' + base64.b64encode(f.read()).decode()

def _html_table(df, caption=None, max_rows=None):
    if df is None or (hasattr(df,'__len__') and len(df)==0):
        return '<p><em>No data</em></p>'
    if max_rows:
        df = df.head(max_rows)
    cap = f"<caption><b>{caption}</b></caption>" if caption else ''
    hdr = ''.join(f'<th>{c}</th>' for c in df.columns)
    body = ''
    for _, row in df.iterrows():
        cells = ''
        for v in row:
            isnan = False
            try: isnan = pd.isna(v)
            except: pass
            if isnan:
                cells += "<td style='color:#aaa'>—</td>"
            elif isinstance(v, (float, np.floating)):
                cells += f'<td>{v:.5g}</td>'
            else:
                cells += f'<td>{v}</td>'
        body += f'<tr>{cells}</tr>'
    return (
        "<table border=1 cellspacing=0 cellpadding=4 "
        "style='border-collapse:collapse;font-size:11px;font-family:monospace;margin:4px 0'>"
        f"{cap}<thead style='background:#f5f5f5'><tr>{hdr}</tr></thead>"
        f"<tbody>{body}</tbody></table>"
    )

def _embed_img(path, caption='', height=280):
    b64 = _img_b64(path)
    if b64 is None:
        return f"<p style='color:#f44336;font-size:11px'>Image not found: {path}</p>"
    return (
        f"<figure style='margin:8px 0;text-align:left'>"
        f"<img src='{b64}' style='height:{height}px;max-width:100%;border:1px solid #ddd;border-radius:4px'>"
        + (f"<figcaption style='font-size:10px;color:#555;margin-top:3px'>{caption}</figcaption>" if caption else '')
        + "</figure>"
    )

def _first_epoch(df, col, thr, direction):
    if col not in df.columns: return None
    for ep, v in zip(df['epoch'], df[col]):
        try:
            if pd.isna(v): continue
        except: pass
        if direction == 'above' and float(v) >= thr: return int(ep)
        if direction == 'below' and float(v) <= thr: return int(ep)
    return None

print('Helpers defined.')

## Section 1 — Run Inventory

In [ ]:
rows = []
for (lr, seed), info in ALL_RUNS.items():
    traj = _get_traj(lr, seed)
    n_epochs = int(traj['epoch'].max()) if traj is not None and 'epoch' in traj.columns else None
    csvs_found = list(dfs.get((lr, seed), {}).keys())
    rows.append({
        'Run': info['name'],
        'LR': lr,
        'Seed': seed,
        'Exists': 'OK' if info['exists'] else 'MISSING',
        'Max epoch': n_epochs,
        'CSVs loaded': len(csvs_found),
    })
inv_df = pd.DataFrame(rows)
display(inv_df)

## Section 2 — Final Performance Across Seeds

In [ ]:
FINAL_COLS = [
    ('network_test_acc', 'Net acc'),
    ('logistic_acc',     'Logistic'),
    ('rbf_svm_acc',      'RBF SVM'),
    ('rf_acc',           'RandForest'),
    ('cka_to_final',     'CKA-to-final'),
    ('local_cka_change', 'Local CKA Δ'),
    ('log10_nc1',        'log10(NC1)'),
    ('nc2_etf_deviation','NC2 ETF dev'),
    ('entk_distance_final','eNTK dist-final'),
]

rows = []
for lr in LR_LIST:
    for seed in SEED_LIST:
        df = _get_traj(lr, seed)
        if df is None: continue
        final = df.iloc[-1]
        row = {'LR': lr, 'Seed': seed}
        for col, lbl in FINAL_COLS:
            v = final.get(col, np.nan)
            try: v = float(v)
            except: v = np.nan
            row[lbl] = round(v, 4) if not np.isnan(v) else np.nan
        rows.append(row)

final_df = pd.DataFrame(rows)
print('=== Per-run final metrics ===')
display(final_df)

# Aggregated summary
agg_rows = []
for lr in LR_LIST:
    sub = final_df[final_df['LR'] == lr]
    row = {'LR': lr}
    for col, lbl in FINAL_COLS:
        s = sub[lbl].dropna()
        row[f'{lbl} mean'] = round(float(s.mean()), 4) if len(s) else np.nan
        row[f'{lbl} std']  = round(float(s.std()),  4) if len(s) > 1 else np.nan
    agg_rows.append(row)
agg_final_df = pd.DataFrame(agg_rows)
print('\n=== Aggregated final metrics (mean ± std across 5 seeds) ===')
display(agg_final_df)

## Section 3 — Primary Evidence: Probes vs CKA

Near-final probe performance is reached substantially earlier than CKA-based stabilization.
This gap is consistent across all three learning rates.

The remaining sections test two complementary explanations. Cross-learning-rate CKA asks
whether different learning rates converge to similar representation geometries. Spectral
component analysis asks whether early probe performance is concentrated in the dominant
singular directions of the representation.

In [ ]:
PROBE_COLS = [('logistic_acc','Logistic'), ('rbf_svm_acc','RBF SVM')]
CKA_COLS   = [('cka_to_final','CKA-to-final'), ('local_cka_change','Local CKA Δ'), ('mean_future_cka','Mean future CKA')]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for row_i, (col, lbl) in enumerate(PROBE_COLS[:2]):
    for col_i, lr in enumerate(LR_LIST):
        ax = axes[row_i][col_i]
        agg_d = _agg_over_seeds(lr, col)
        if agg_d.empty:
            ax.set_visible(False); continue
        final_val = agg_d['mean'].iloc[-1]
        ax.plot(agg_d['epoch'], agg_d['mean'], color=LR_COLORS[lr], lw=2, label=lbl)
        ax.fill_between(agg_d['epoch'], agg_d['mean']-agg_d['std'], agg_d['mean']+agg_d['std'], alpha=0.2, color=LR_COLORS[lr])
        ax.axhline(final_val * 0.95, color='#e65100', ls='--', lw=1.2, label='95% of final probe')
        ax.axhline(final_val,        color='gray',    ls=':',  lw=1.0, label='Final probe value')
        ax.set_title(f'{lbl} — LR={lr}', fontsize=10, fontweight='bold')
        ax.set_xlabel('Epoch', fontsize=8); ax.set_ylabel('Test accuracy', fontsize=8)
        ax.legend(fontsize=7)

fig.suptitle('Probe Trajectories: mean ± std across 5 seeds', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# CKA trajectories
fig2, axes2 = plt.subplots(1, 3, figsize=(15, 4))
for i, lr in enumerate(LR_LIST):
    ax = axes2[i]
    for col, lbl in CKA_COLS:
        agg_d = _agg_over_seeds(lr, col)
        if agg_d.empty: continue
        ax.plot(agg_d['epoch'], agg_d['mean'], lw=1.8, label=lbl)
        ax.fill_between(agg_d['epoch'], agg_d['mean']-agg_d['std'], agg_d['mean']+agg_d['std'], alpha=0.15)
    ax.axhline(0.02, color='#b71c1c', ls='--', lw=1, label='τ=0.02 (local Δ)')
    ax.set_title(f'CKA Metrics — LR={lr}', fontsize=10, fontweight='bold')
    ax.set_xlabel('Epoch', fontsize=8); ax.legend(fontsize=7)

fig2.suptitle('CKA Trajectories: mean ± std across 5 seeds', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Section 4 — Learning-Rate Comparison

In [ ]:
COMPARE_COLS = [
    ('network_test_acc',  'Network test acc'),
    ('logistic_acc',      'Logistic probe acc'),
    ('rbf_svm_acc',       'RBF SVM probe acc'),
    ('cka_to_final',      'CKA to final'),
    ('mean_future_cka',   'Mean future CKA'),
    ('local_cka_change',  'Local CKA Δ'),
    ('log10_nc1',         'log10(NC1)'),
    ('nc2_etf_deviation', 'NC2 ETF deviation'),
    ('entk_distance_final','eNTK dist-to-final'),
]

ncols = 3
nrows = (len(COMPARE_COLS) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows*3.5))
axes_flat = axes.flatten()

for i, (col, lbl) in enumerate(COMPARE_COLS):
    ax = axes_flat[i]
    for lr in LR_LIST:
        agg_d = _agg_over_seeds(lr, col)
        if agg_d.empty: continue
        ax.plot(agg_d['epoch'], agg_d['mean'], color=LR_COLORS[lr], lw=1.8, label=f'LR={lr}')
        ax.fill_between(agg_d['epoch'], agg_d['mean']-agg_d['std'], agg_d['mean']+agg_d['std'], alpha=0.15, color=LR_COLORS[lr])
    ax.set_title(lbl, fontsize=10, fontweight='bold')
    ax.set_xlabel('Epoch', fontsize=8)
    ax.legend(fontsize=7)

for i in range(len(COMPARE_COLS), len(axes_flat)):
    axes_flat[i].set_visible(False)

fig.suptitle('Learning-Rate Comparison — mean ± std across 5 seeds', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Section 5 — Within-LR Seed Robustness

In [ ]:
ROBUST_COLS = [('logistic_acc','Logistic acc'), ('cka_to_final','CKA to final')]

fig, axes = plt.subplots(len(LR_LIST), len(ROBUST_COLS), figsize=(12, 10))
SEED_COLORS = plt.cm.tab10(np.linspace(0, 0.5, len(SEED_LIST)))

for row_i, lr in enumerate(LR_LIST):
    for col_i, (col, lbl) in enumerate(ROBUST_COLS):
        ax = axes[row_i][col_i]
        for j, seed in enumerate(SEED_LIST):
            df = _get_traj(lr, seed)
            if df is None or col not in df.columns: continue
            ax.plot(df['epoch'], df[col], color=SEED_COLORS[j], lw=1.3, alpha=0.85, label=f'seed={seed}')
        ax.set_title(f'{lbl} — LR={lr}', fontsize=10, fontweight='bold')
        ax.set_xlabel('Epoch', fontsize=8)
        ax.legend(fontsize=7)

fig.suptitle('Within-LR Seed Robustness (individual seed traces)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Coefficient of variation at final epoch
cv_rows = []
for lr in LR_LIST:
    for col, lbl in [('logistic_acc','Logistic'),('cka_to_final','CKA-to-final'),('network_test_acc','Net acc')]:
        vals = []
        for seed in SEED_LIST:
            df = _get_traj(lr, seed)
            if df is not None and col in df.columns:
                v = df[col].dropna().iloc[-1] if not df[col].dropna().empty else np.nan
                vals.append(float(v))
        if vals:
            cv_rows.append({'LR': lr, 'Metric': lbl,
                            'Mean': round(np.mean(vals),4), 'Std': round(np.std(vals),4),
                            'CV%': round(100*np.std(vals)/np.mean(vals),2) if np.mean(vals)!=0 else np.nan})
print('\n=== Final-epoch CV across seeds ===')
display(pd.DataFrame(cv_rows))

## Section 6 — Cross-Learning-Rate CKA

Final probe and network accuracies are similar across learning rates, so cross-LR CKA asks
whether the representations also converge geometrically, or whether different learning rates
reach similar performance through different representation geometries.

High same-epoch CKA would indicate convergent geometry at the same training progress;
lower final-epoch cross-LR CKA compared to within-LR cross-seed similarity would indicate
that different learning rates leave distinct representational fingerprints even at convergence.

In [ ]:
# ── Same-epoch cross-LR CKA trajectory ────────────────────────────────────────
cross_lr_png = AGG_DIR / 'plots' / 'cross_lr_cka' / 'cross_lr_cka_same_epoch_trajectory.png'

if cross_lr_png.exists():
    fig, ax = plt.subplots(figsize=(10, 4))
    img = plt.imread(str(cross_lr_png))
    ax.imshow(img); ax.axis('off')
    ax.set_title('Cross-LR same-epoch CKA trajectory (pre-generated)', fontsize=11)
    plt.tight_layout(); plt.show()
elif agg.get('cross_lr_same_epoch') is not None:
    df_se = agg['cross_lr_same_epoch']
    pairs = df_se[['lr_a','lr_b']].drop_duplicates().values.tolist()
    pair_colors = ['#9c27b0','#e91e63','#ff5722']
    fig, ax = plt.subplots(figsize=(10, 4))
    for (la, lb), col in zip(pairs, pair_colors):
        sub = df_se[(df_se['lr_a']==la) & (df_se['lr_b']==lb)]
        lbl = f'LR={la} vs LR={lb}'
        ax.plot(sub['epoch'], sub['mean_cka'], color=col, lw=1.8, label=lbl)
        ax.fill_between(sub['epoch'], sub['mean_cka']-sub['std_cka'], sub['mean_cka']+sub['std_cka'], alpha=0.15, color=col)
    ax.set_xlabel('Epoch', fontsize=10)
    ax.set_ylabel('CKA similarity', fontsize=10)
    ax.set_title('Same-epoch cross-LR CKA (mean ± std over seed-matched pairs)', fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)
    plt.tight_layout(); plt.show()
else:
    print('WARNING: cross-LR same-epoch data not available.')

In [ ]:
# ── Final-epoch CKA matrix ────────────────────────────────────────────────────
matrix_png = AGG_DIR / 'plots' / 'cross_lr_cka' / 'final_epoch_all_runs_cka_matrix.png'
if matrix_png.exists():
    fig, ax = plt.subplots(figsize=(8, 7))
    img = plt.imread(str(matrix_png))
    ax.imshow(img); ax.axis('off')
    ax.set_title('Final-epoch CKA matrix across all 15 runs (pre-generated)', fontsize=11)
    plt.tight_layout(); plt.show()
elif agg.get('final_cka_matrix') is not None:
    mat = agg['final_cka_matrix'].set_index('run_name')
    fig, ax = plt.subplots(figsize=(9, 8))
    im = ax.imshow(mat.values.astype(float), vmin=0.93, vmax=1.0, cmap='RdYlGn')
    ax.set_xticks(range(len(mat.columns))); ax.set_xticklabels(mat.columns, rotation=90, fontsize=7)
    ax.set_yticks(range(len(mat.index)));   ax.set_yticklabels(mat.index, fontsize=7)
    plt.colorbar(im, ax=ax, label='CKA similarity')
    ax.set_title('Final-epoch CKA similarity matrix (all 15 runs)', fontsize=11, fontweight='bold')
    plt.tight_layout(); plt.show()
else:
    print('WARNING: final CKA matrix not available.')

# ── Summary bar chart ─────────────────────────────────────────────────────────
bar_png = AGG_DIR / 'plots' / 'cross_lr_cka' / 'final_similarity_summary_bar.png'
if bar_png.exists():
    fig, ax = plt.subplots(figsize=(10, 4))
    img = plt.imread(str(bar_png))
    ax.imshow(img); ax.axis('off')
    ax.set_title('Final similarity summary (pre-generated)', fontsize=11)
    plt.tight_layout(); plt.show()
elif agg.get('combined_final_sim') is not None:
    cfs = agg['combined_final_sim']
    fig, ax = plt.subplots(figsize=(10, 4))
    x = range(len(cfs))
    bars = ax.bar(x, cfs['mean_final_cka'], yerr=cfs['std_final_cka'], capsize=4,
                  color=['#1f77b4','#ff7f0e','#2ca02c','#9c27b0','#e91e63','#ff5722'][:len(cfs)])
    ax.set_xticks(x)
    ax.set_xticklabels(cfs['comparison'], rotation=25, ha='right', fontsize=9)
    ax.set_ylabel('Mean final CKA', fontsize=10)
    ax.set_ylim(0.93, 1.0)
    ax.set_title('Final-epoch CKA: within-LR cross-seed vs cross-LR similarity', fontsize=11, fontweight='bold')
    plt.tight_layout(); plt.show()

# Summary table
if agg.get('combined_final_sim') is not None:
    print('\n=== Final-epoch CKA similarity summary ===')
    display(agg['combined_final_sim'].round(5))

In [ ]:
# ── Mean epoch-by-epoch heatmaps ──────────────────────────────────────────────
heatmap_pairs = [
    ('mean_heatmap_lr005_vs_lr010.png', 'LR=0.05 vs LR=0.10'),
    ('mean_heatmap_lr005_vs_lr020.png', 'LR=0.05 vs LR=0.20'),
    ('mean_heatmap_lr010_vs_lr020.png', 'LR=0.10 vs LR=0.20'),
]
hm_dir = AGG_DIR / 'plots' / 'cross_lr_cka'

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (fname, title) in zip(axes, heatmap_pairs):
    fp = hm_dir / fname
    if fp.exists():
        img = plt.imread(str(fp))
        ax.imshow(img); ax.axis('off')
    else:
        ax.text(0.5, 0.5, f'Missing:\n{fname}', ha='center', va='center', transform=ax.transAxes, color='red')
        ax.axis('off')
    ax.set_title(title, fontsize=10, fontweight='bold')

fig.suptitle(
    'Epoch-by-epoch cross-LR CKA heatmaps (mean over seeds).\n'
    'Rows = epochs of LR_a, columns = epochs of LR_b. '
    'Off-diagonal ridges indicate different learning speeds.',
    fontsize=10
)
plt.tight_layout()
plt.show()

## Section 7 — Spectral Component Analysis

This is a post-hoc diagnostic inspired by Deng et al. At each checkpoint the penultimate
feature matrix is decomposed with SVD (fitted on training features only; test features are
projected using the train-fitted basis). The **main component** retains the smallest number
of leading singular directions whose cumulative singular-value mass reaches 80%; the
**residual component** contains the remaining directions. Logistic regression probes are
then trained on the full, main, and residual feature sets.

The goal is to identify whether early readout-accessible performance is concentrated in
dominant singular directions, consistent with the interpretation that class information
becomes concentrated in a low-dimensional subspace before the representation geometry
finishes evolving.

In [ ]:
# ── Probe trajectories: full / main / residual ────────────────────────────────
if spectral_pngs.get('probe_trajectories') and spectral_pngs['probe_trajectories'].exists():
    fig, ax = plt.subplots(figsize=(15, 4))
    img = plt.imread(str(spectral_pngs['probe_trajectories']))
    ax.imshow(img); ax.axis('off')
    plt.tight_layout(); plt.show()
else:
    print('WARNING: spectral probe trajectory PNG not found. Re-run cell-spectral-pngs.')

# k_main and spectral mass
for key, title in [('k_main_over_time','k_main Over Time'), ('spectral_mass_over_time','Spectral Mass Over Time')]:
    p = spectral_pngs.get(key)
    if p and p.exists():
        fig, ax = plt.subplots(figsize=(9, 4))
        ax.imshow(plt.imread(str(p))); ax.axis('off')
        plt.tight_layout(); plt.show()
    else:
        print(f'WARNING: {key} PNG not found.')

In [ ]:
# ── Spectral final-performance table ──────────────────────────────────────────
if agg.get('spectral_probes') is not None and agg.get('spectral_summary') is not None:
    sp  = agg['spectral_probes'][agg['spectral_probes']['probe'] == 'logistic_regression'].copy()
    ss  = agg['spectral_summary'].copy()

    # Final epoch per run
    final_ep = sp.groupby(['lr','seed'])['epoch'].max().reset_index().rename(columns={'epoch':'final_epoch'})
    sp_final = sp.merge(final_ep, on=['lr','seed'])
    sp_final = sp_final[sp_final['epoch'] == sp_final['final_epoch']]

    ss_final = ss.merge(final_ep, on=['lr','seed'])
    ss_final = ss_final[ss_final['epoch'] == ss_final['final_epoch']]

    rows = []
    for lr_f in sorted(sp['lr'].unique()):
        lr_str = f'{lr_f:.2f}'
        for comp in ['full','main','residual']:
            sub_acc = sp_final[(sp_final['lr']==lr_f) & (sp_final['component']==comp)]['test_acc']
            sub_ss  = ss_final[ss_final['lr']==lr_f]
            rows.append({
                'LR': lr_str, 'Component': comp,
                'Final probe acc (mean)': round(sub_acc.mean(),4) if len(sub_acc) else np.nan,
                'Final probe acc (std)':  round(sub_acc.std(), 4) if len(sub_acc)>1 else np.nan,
                'k_main (mean)':          round(sub_ss['k_main'].mean(),1) if len(sub_ss) else np.nan,
                'main sing. mass (mean)': round(sub_ss['main_singular_mass'].mean(),4) if len(sub_ss) else np.nan,
                'main energy mass (mean)':round(sub_ss['main_energy_mass'].mean(),4) if len(sub_ss) else np.nan,
            })

    spec_final_df = pd.DataFrame(rows)
    print('=== Spectral final-performance table ===')
    display(spec_final_df)

    # Save for later use in HTML dashboard
    out_csv = AGG_DIR / 'spectral_final_performance_by_lr_component.csv'
    spec_final_df.to_csv(out_csv, index=False)
    print(f'Saved: {out_csv}')
else:
    spec_final_df = None
    print('WARNING: spectral data not available for final-performance table.')

In [ ]:
# ── Spectral operational timing table ─────────────────────────────────────────
# First epoch where component probe reaches 95% of its own final-epoch accuracy.
if agg.get('spectral_probes') is not None:
    sp = agg['spectral_probes'][agg['spectral_probes']['probe'] == 'logistic_regression'].copy()
    timing_rows = []
    for lr_f in sorted(sp['lr'].unique()):
        lr_str = f'{lr_f:.2f}'
        sub_lr = sp[sp['lr'] == lr_f]
        for comp in ['full','main','residual']:
            sub_comp = sub_lr[sub_lr['component'] == comp]
            epochs_hit = []
            for seed in sub_comp['seed'].unique():
                sub_seed = sub_comp[sub_comp['seed'] == seed].sort_values('epoch')
                if sub_seed.empty: continue
                final_acc = float(sub_seed['test_acc'].iloc[-1])
                thr = 0.95 * final_acc
                hit = None
                for _, row in sub_seed.iterrows():
                    if float(row['test_acc']) >= thr:
                        hit = int(row['epoch']); break
                if hit is not None:
                    epochs_hit.append(hit)
            timing_rows.append({
                'LR': lr_str, 'Component': comp,
                'Mean epoch (95% of final)': round(np.mean(epochs_hit),1) if epochs_hit else np.nan,
                'Std epoch':                 round(np.std(epochs_hit),1)  if len(epochs_hit)>1 else np.nan,
                'n': len(epochs_hit),
            })

    spec_timing_df = pd.DataFrame(timing_rows)
    print('=== Spectral operational timing (first epoch reaching 95% of final component probe acc) ===')
    display(spec_timing_df)
    out_csv2 = AGG_DIR / 'spectral_timing_events_by_lr_component.csv'
    spec_timing_df.to_csv(out_csv2, index=False)
    print(f'Saved: {out_csv2}')
else:
    spec_timing_df = None
    print('WARNING: spectral probe data not available for timing table.')

## Section 8 — Neural Collapse

In [ ]:
NC_COLS = [
    ('log10_nc1', 'log10(NC1) — within-class variability'),
    ('nc2_etf_deviation', 'NC2 — ETF deviation'),
    ('nc3_weight_mean_alignment', 'NC3 — weight-mean alignment'),
    ('nc4_ncm_disagreement', 'NC4 — NCM disagreement'),
]

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
axes_flat = axes.flatten()

for i, (col, lbl) in enumerate(NC_COLS):
    ax = axes_flat[i]
    for lr in LR_LIST:
        agg_d = _agg_over_seeds(lr, col)
        if agg_d.empty: continue
        ax.plot(agg_d['epoch'], agg_d['mean'], color=LR_COLORS[lr], lw=1.8, label=f'LR={lr}')
        ax.fill_between(agg_d['epoch'], agg_d['mean']-agg_d['std'], agg_d['mean']+agg_d['std'], alpha=0.15, color=LR_COLORS[lr])
    ax.set_title(lbl, fontsize=10, fontweight='bold')
    ax.set_xlabel('Epoch', fontsize=8)
    ax.legend(fontsize=8)

fig.suptitle('Neural Collapse Metrics — mean ± std across 5 seeds', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Section 9 — Last-Layer eNTK

In [ ]:
ENTK_COLS = [
    ('entk_distance_final', 'eNTK distance to final'),
    ('mean_future_entk_similarity', 'Mean future eNTK similarity'),
    ('entk_distance_prev', 'eNTK distance to previous'),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, (col, lbl) in enumerate(ENTK_COLS):
    ax = axes[i]
    for lr in LR_LIST:
        agg_d = _agg_over_seeds(lr, col)
        if agg_d.empty: continue
        ax.plot(agg_d['epoch'], agg_d['mean'], color=LR_COLORS[lr], lw=1.8, label=f'LR={lr}')
        ax.fill_between(agg_d['epoch'], agg_d['mean']-agg_d['std'], agg_d['mean']+agg_d['std'], alpha=0.15, color=LR_COLORS[lr])
    ax.set_title(lbl, fontsize=10, fontweight='bold')
    ax.set_xlabel('Epoch', fontsize=8)
    ax.legend(fontsize=8)

fig.suptitle('eNTK Dynamics — mean ± std across 5 seeds', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Section 10 — Operational Timing Summaries

In [ ]:
TIMING_EVENTS = [
    ('logistic_acc',            None, 0.95, 'above', 'Logistic probe: 95% of final'),
    ('logistic_acc',            None, 0.99, 'above', 'Logistic probe: 99% of final'),
    ('rbf_svm_acc',             None, 0.95, 'above', 'RBF SVM probe: 95% of final'),
    ('cka_to_final',            0.95, None, 'above', 'CKA-to-final ≥ 0.95'),
    ('cka_to_final',            0.99, None, 'above', 'CKA-to-final ≥ 0.99'),
    ('local_cka_change',        None, 0.02, 'below', 'Local CKA Δ ≤ 0.02'),
    ('entk_distance_final',     None, 0.05, 'below', 'eNTK dist-to-final ≤ 0.05'),
    ('entk_distance_final',     None, 0.01, 'below', 'eNTK dist-to-final ≤ 0.01'),
    ('log10_nc1',               None, -1.0, 'below', 'log10(NC1) ≤ -1'),
]

timing_rows = []
for lr in LR_LIST:
    for col, abs_thr, frac_thr, direction, label in TIMING_EVENTS:
        epochs_hit = []
        for seed in SEED_LIST:
            df = _get_traj(lr, seed)
            if df is None or col not in df.columns: continue
            series = df[col].dropna()
            if series.empty: continue
            if frac_thr is not None:
                thr = float(series.iloc[-1]) * frac_thr
            else:
                thr = abs_thr
            ep = _first_epoch(df, col, thr, direction)
            if ep is not None:
                epochs_hit.append(ep)
        timing_rows.append({
            'LR': lr,
            'Event': label,
            'Mean epoch': round(np.mean(epochs_hit),1) if epochs_hit else np.nan,
            'Std epoch':  round(np.std(epochs_hit),1)  if len(epochs_hit)>1 else np.nan,
            'n': len(epochs_hit),
        })

timing_df = pd.DataFrame(timing_rows)
print('=== Operational timing: mean epoch across seeds ===')
print('(95%/99% probe thresholds are relative to each probe\'s own final-epoch value)')
display(timing_df.pivot(index='Event', columns='LR', values='Mean epoch'))

## Export HTML Dashboard

Run the cell below to write `notebooks/.generated/dashboard.html`.

In [ ]:
# ── HTML dashboard builder ────────────────────────────────────────────────────

SEC  = ("background:#1a1a2e;color:#e0e0e0;padding:10px 20px;margin:24px 0 6px 0;"
        "font-family:monospace;font-size:15px;letter-spacing:1px;border-left:4px solid #4fc3f7")
SUB  = ("background:#f0f4ff;color:#1a237e;padding:5px 14px;margin:10px 0 4px 0;"
        "font-family:monospace;font-size:12px;border-left:3px solid #3949ab;font-weight:bold")
NOTE = ("font-size:11px;color:#444;font-family:monospace;background:#fffde7;"
        "border-left:3px solid #f9a825;padding:6px 12px;margin:6px 0")
PROSE= "font-size:12px;color:#333;font-family:sans-serif;line-height:1.5;margin:6px 0 10px 0"

def _section(title): return f"<div style='{SEC}'>{title}</div>"
def _sub(title):     return f"<div style='{SUB}'>{title}</div>"
def _note(text):     return f"<div style='{NOTE}'>{text}</div>"
def _prose(text):    return f"<p style='{PROSE}'>{text}</p>"

def _make_probe_cka_fig_b64():
    PROBE_COLS_D = [('logistic_acc','Logistic'), ('rbf_svm_acc','RBF SVM')]
    CKA_COLS_D   = [('cka_to_final','CKA-to-final'), ('local_cka_change','Local CKA Δ')]
    ncols, nrows = 3, len(PROBE_COLS_D) + 1
    fig, axes = plt.subplots(nrows, ncols, figsize=(15, nrows*3.2))
    for row_i, (col, lbl) in enumerate(PROBE_COLS_D):
        for col_i, lr in enumerate(LR_LIST):
            ax = axes[row_i][col_i]
            agg_d = _agg_over_seeds(lr, col)
            if agg_d.empty: ax.set_visible(False); continue
            fv = agg_d['mean'].iloc[-1]
            ax.plot(agg_d['epoch'], agg_d['mean'], color=LR_COLORS[lr], lw=2, label=lbl)
            ax.fill_between(agg_d['epoch'], agg_d['mean']-agg_d['std'], agg_d['mean']+agg_d['std'], alpha=0.18, color=LR_COLORS[lr])
            ax.axhline(fv*0.95, color='#e65100', ls='--', lw=1.1, label='95% of final probe')
            ax.set_title(f'{lbl} — LR={lr}', fontsize=9, fontweight='bold')
            ax.set_xlabel('Epoch',8); ax.legend(fontsize=7)
    for col_i, lr in enumerate(LR_LIST):
        ax = axes[nrows-1][col_i]
        for col2, lbl2 in CKA_COLS_D:
            agg_d = _agg_over_seeds(lr, col2)
            if agg_d.empty: continue
            ax.plot(agg_d['epoch'], agg_d['mean'], lw=1.8, label=lbl2)
            ax.fill_between(agg_d['epoch'], agg_d['mean']-agg_d['std'], agg_d['mean']+agg_d['std'], alpha=0.13)
        ax.axhline(0.02, color='#b71c1c', ls='--', lw=1, label='τ=0.02')
        ax.set_title(f'CKA — LR={lr}', fontsize=9, fontweight='bold')
        ax.set_xlabel('Epoch', fontsize=8); ax.legend(fontsize=7)
    fig.suptitle('Probes vs CKA — mean ± std across 5 seeds', fontsize=12, fontweight='bold')
    plt.tight_layout()
    return _fig_to_b64(fig)

def _make_lr_comparison_fig_b64():
    cols = [('logistic_acc','Logistic acc'),('rbf_svm_acc','RBF SVM acc'),
            ('cka_to_final','CKA-to-final'),('local_cka_change','Local CKA Δ'),
            ('log10_nc1','log10(NC1)'),('entk_distance_final','eNTK dist-final')]
    fig, axes = plt.subplots(2, 3, figsize=(15, 7))
    for ax, (col, lbl) in zip(axes.flatten(), cols):
        for lr in LR_LIST:
            agg_d = _agg_over_seeds(lr, col)
            if agg_d.empty: continue
            ax.plot(agg_d['epoch'], agg_d['mean'], color=LR_COLORS[lr], lw=1.8, label=f'LR={lr}')
            ax.fill_between(agg_d['epoch'], agg_d['mean']-agg_d['std'], agg_d['mean']+agg_d['std'], alpha=0.15, color=LR_COLORS[lr])
        ax.set_title(lbl, fontsize=10, fontweight='bold'); ax.set_xlabel('Epoch',8); ax.legend(fontsize=7)
    fig.suptitle('LR Comparison — mean ± std across 5 seeds', fontsize=12, fontweight='bold')
    plt.tight_layout()
    return _fig_to_b64(fig)

def _make_cross_lr_fig_b64():
    if agg.get('cross_lr_same_epoch') is None: return None
    df_se = agg['cross_lr_same_epoch']
    pairs = df_se[['lr_a','lr_b']].drop_duplicates().values.tolist()
    pair_colors = ['#9c27b0','#e91e63','#ff5722']
    fig, ax = plt.subplots(figsize=(9, 4))
    for (la, lb), col in zip(pairs, pair_colors):
        sub = df_se[(df_se['lr_a']==la) & (df_se['lr_b']==lb)]
        ax.plot(sub['epoch'], sub['mean_cka'], color=col, lw=1.8, label=f'LR={la} vs {lb}')
        ax.fill_between(sub['epoch'], sub['mean_cka']-sub['std_cka'], sub['mean_cka']+sub['std_cka'], alpha=0.15, color=col)
    ax.set_xlabel('Epoch',10); ax.set_ylabel('CKA similarity',10)
    ax.set_title('Same-epoch cross-LR CKA (mean ± std, seed-matched pairs)', fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)
    plt.tight_layout()
    return _fig_to_b64(fig)

def _make_spectral_probes_fig_b64():
    if agg.get('spectral_probes') is None: return None
    sp = agg['spectral_probes'][agg['spectral_probes']['probe']=='logistic_regression'].copy()
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
    comp_style = [('full','#1a237e','-'),('main','#2e7d32','--'),('residual','#b71c1c',':')]
    for i, lr in enumerate(LR_LIST):
        ax = axes[i]
        sub = sp[sp['lr']==float(lr)]
        for comp, color, ls in comp_style:
            cs = sub[sub['component']==comp].groupby('epoch')['test_acc'].agg(['mean','std']).reset_index()
            if cs.empty: continue
            ax.plot(cs['epoch'], cs['mean'], color=color, ls=ls, lw=1.8, label=comp)
            ax.fill_between(cs['epoch'], cs['mean']-cs['std'], cs['mean']+cs['std'], alpha=0.15, color=color)
        ax.set_title(f'LR = {lr}', fontsize=11, fontweight='bold')
        ax.set_xlabel('Epoch', fontsize=9)
        if i==0: ax.set_ylabel('Logistic probe test accuracy', fontsize=9)
        ax.legend(fontsize=8); ax.set_ylim(0,1)
    fig.suptitle('Spectral Component Probe Trajectories (mean ± std across seeds)', fontsize=12, fontweight='bold')
    plt.tight_layout()
    return _fig_to_b64(fig)

def _make_nc_fig_b64():
    cols = [('log10_nc1','log10(NC1)'),('nc2_etf_deviation','NC2 ETF dev'),
            ('nc3_weight_mean_alignment','NC3 alignment'),('nc4_ncm_disagreement','NC4 NCM disagr.')]
    fig, axes = plt.subplots(2, 2, figsize=(12, 7))
    for ax, (col, lbl) in zip(axes.flatten(), cols):
        for lr in LR_LIST:
            agg_d = _agg_over_seeds(lr, col)
            if agg_d.empty: continue
            ax.plot(agg_d['epoch'], agg_d['mean'], color=LR_COLORS[lr], lw=1.8, label=f'LR={lr}')
            ax.fill_between(agg_d['epoch'], agg_d['mean']-agg_d['std'], agg_d['mean']+agg_d['std'], alpha=0.15, color=LR_COLORS[lr])
        ax.set_title(lbl, fontsize=10, fontweight='bold'); ax.set_xlabel('Epoch',8); ax.legend(fontsize=8)
    fig.suptitle('Neural Collapse — mean ± std across 5 seeds', fontsize=12, fontweight='bold')
    plt.tight_layout()
    return _fig_to_b64(fig)

def _make_entk_fig_b64():
    cols = [('entk_distance_final','eNTK dist-to-final'),
            ('mean_future_entk_similarity','Mean future eNTK sim'),
            ('entk_distance_prev','eNTK dist-to-prev')]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, (col, lbl) in zip(axes, cols):
        for lr in LR_LIST:
            agg_d = _agg_over_seeds(lr, col)
            if agg_d.empty: continue
            ax.plot(agg_d['epoch'], agg_d['mean'], color=LR_COLORS[lr], lw=1.8, label=f'LR={lr}')
            ax.fill_between(agg_d['epoch'], agg_d['mean']-agg_d['std'], agg_d['mean']+agg_d['std'], alpha=0.15, color=LR_COLORS[lr])
        ax.set_title(lbl, fontsize=10, fontweight='bold'); ax.set_xlabel('Epoch',8); ax.legend(fontsize=8)
    fig.suptitle('eNTK Dynamics — mean ± std across 5 seeds', fontsize=12, fontweight='bold')
    plt.tight_layout()
    return _fig_to_b64(fig)

print('Dashboard builder functions defined.')

In [ ]:
def build_dashboard(save_html=True):
    p = []

    # ── Header ────────────────────────────────────────────────────────────────
    p.append(
        "<div style='font-family:monospace;background:#0d0d1a;padding:20px 28px;"
        "border-radius:6px;margin-bottom:16px'>"
        "<h1 style='color:#4fc3f7;margin:0;font-size:20px;letter-spacing:2px'>"
        "MULTI-SEED RESULTS DASHBOARD</h1>"
        "<p style='color:#78909c;margin:6px 0 0 0;font-size:11px'>"
        "ResNet-18 &nbsp;·&nbsp; CIFAR-10 &nbsp;·&nbsp; SGD+CosineAnnealing"
        " &nbsp;·&nbsp; 200 epochs &nbsp;·&nbsp; 3 LRs × 5 seeds = 15 runs<br>"
        "<b style='color:#b0bec5'>Question:</b> "
        "Does representation stabilization imply representation sufficiency?</p></div>"
    )

    # ── 1. Run inventory ──────────────────────────────────────────────────────
    p.append(_section('1. RUN INVENTORY'))
    rows_inv = []
    for (lr, seed), info in ALL_RUNS.items():
        traj = _get_traj(lr, seed)
        ne = int(traj['epoch'].max()) if traj is not None and 'epoch' in traj.columns else None
        rows_inv.append({'Run': info['name'], 'LR': lr, 'Seed': seed,
                         'Status': 'OK' if info['exists'] else 'MISSING',
                         'Max epoch': ne, 'CSVs': len(dfs.get((lr,seed),{}))})
    p.append(_html_table(pd.DataFrame(rows_inv), 'All 15 runs'))

    # ── 2. Final performance ──────────────────────────────────────────────────
    p.append(_section('2. FINAL PERFORMANCE ACROSS SEEDS'))
    p.append(_html_table(final_df, 'Per-run final metrics'))
    p.append(_html_table(agg_final_df, 'Aggregated final metrics (mean ± std across 5 seeds)'))

    # ── 3. Primary evidence: probes vs CKA ───────────────────────────────────
    p.append(_section('3. PRIMARY EVIDENCE: PROBES vs CKA'))
    p.append(_prose(
        'Near-final probe performance is reached substantially earlier than CKA-based stabilization '
        'across all three learning rates and all five seeds. The logistic probe in particular '
        'reaches 95% of its final-epoch accuracy well before the local CKA change falls below '
        'the &tau;=0.02 threshold, consistent with the interpretation that readout-accessible '
        'class information matures before representation geometry stabilizes. '
        'The remaining sections test two complementary explanations. '
        'Cross-learning-rate CKA asks whether different learning rates converge to similar '
        'representation geometries. Spectral component analysis asks whether early probe '
        'performance is concentrated in the dominant singular directions of the representation.'
    ))
    b64 = _make_probe_cka_fig_b64()
    if b64: p.append(f"<img src='{b64}' style='max-width:100%;border:1px solid #ddd;border-radius:4px'>")
    p.append(_html_table(timing_df.pivot(index='Event', columns='LR', values='Mean epoch').reset_index(),
                         'Operational timing: mean epoch across seeds (probe thresholds are relative to each probe\'s final value)'))

    # ── 4. LR comparison ──────────────────────────────────────────────────────
    p.append(_section('4. LEARNING-RATE COMPARISON'))
    b64 = _make_lr_comparison_fig_b64()
    if b64: p.append(f"<img src='{b64}' style='max-width:100%;border:1px solid #ddd;border-radius:4px'>")

    # ── 5. Seed robustness ────────────────────────────────────────────────────
    p.append(_section('5. WITHIN-LR SEED ROBUSTNESS'))
    if agg.get('within_lr_final') is not None:
        p.append(_html_table(agg['within_lr_final'].round(5), 'Within-LR cross-seed final CKA similarity'))
    cv_df2 = pd.DataFrame(cv_rows) if 'cv_rows' in dir() else None
    if cv_df2 is not None: p.append(_html_table(cv_df2, 'Coefficient of variation at final epoch'))

    # ── 6. Cross-LR CKA ───────────────────────────────────────────────────────
    p.append(_section('6. CROSS-LEARNING-RATE CKA'))
    p.append(_prose(
        'Final probe and network accuracies are similar across learning rates. '
        'Cross-LR CKA tests whether representations also converge geometrically, or whether '
        'different learning rates reach similar performance through different representation geometries. '
        '<b>High same-epoch CKA</b> indicates convergent geometry at the same training progress. '
        '<b>Lower final-epoch cross-LR CKA</b> compared to within-LR cross-seed similarity would '
        'indicate that different learning rates leave distinct representational fingerprints even at convergence. '
        '<b>Off-diagonal ridges</b> in the heatmaps indicate different-speed convergence: LR pairs '
        'can reach similar geometry at different epoch counts.'
    ))

    p.append(_sub('Same-epoch cross-LR CKA trajectory'))
    clr_b64 = _make_cross_lr_fig_b64()
    clr_img = _img_b64(AGG_DIR / 'plots' / 'cross_lr_cka' / 'cross_lr_cka_same_epoch_trajectory.png')
    src = clr_img or clr_b64
    if src: p.append(f"<img src='{src}' style='max-width:100%;max-height:320px;border:1px solid #ddd;border-radius:4px'>")
    else:   p.append('<p style=\'color:gray\'>Same-epoch cross-LR CKA trajectory not available.</p>')

    p.append(_sub('Final-epoch CKA similarity summary'))
    if agg.get('combined_final_sim') is not None:
        p.append(_html_table(agg['combined_final_sim'].round(5),
                             'Final-epoch CKA: within-LR cross-seed vs cross-LR (seed-matched)'))
    bar_img = _img_b64(AGG_DIR / 'plots' / 'cross_lr_cka' / 'final_similarity_summary_bar.png')
    if bar_img: p.append(f"<img src='{bar_img}' style='max-height:280px;border:1px solid #ddd;border-radius:4px'>")
    if agg.get('cross_lr_final') is not None:
        p.append(_html_table(agg['cross_lr_final'].round(5), 'Cross-LR final-epoch CKA (seed-matched pairs)'))

    p.append(_sub('Final-epoch all-runs CKA matrix'))
    mat_img = _img_b64(AGG_DIR / 'plots' / 'cross_lr_cka' / 'final_epoch_all_runs_cka_matrix.png')
    if mat_img: p.append(f"<img src='{mat_img}' style='max-height:380px;border:1px solid #ddd;border-radius:4px'>")
    else: p.append('<p style=\'color:gray\'>CKA matrix image not available.</p>')

    p.append(_sub('Epoch-by-epoch cross-LR heatmaps (mean over seeds)'))
    p.append(_note('Rows = epoch of LR_a, columns = epoch of LR_b. '
                   'Off-diagonal ridges indicate different learning speeds.'))
    hm_dir2 = AGG_DIR / 'plots' / 'cross_lr_cka'
    for fname, caption in [
        ('mean_heatmap_lr005_vs_lr010.png','LR=0.05 vs LR=0.10'),
        ('mean_heatmap_lr005_vs_lr020.png','LR=0.05 vs LR=0.20'),
        ('mean_heatmap_lr010_vs_lr020.png','LR=0.10 vs LR=0.20'),
    ]:
        img = _img_b64(hm_dir2 / fname)
        if img:
            p.append(f"<figure style='display:inline-block;margin:6px 12px 6px 0'>"
                     f"<img src='{img}' style='height:260px;border:1px solid #ddd;border-radius:4px'>"
                     f"<figcaption style='font-size:10px;color:#555;text-align:center'>{caption}</figcaption>"
                     f"</figure>")
        else:
            p.append(f"<span style='color:gray;font-size:11px'>Missing: {fname}</span>")

    # ── 7. Spectral component analysis ────────────────────────────────────────
    p.append(_section('7. SPECTRAL COMPONENT ANALYSIS'))
    p.append(_prose(
        'Post-hoc diagnostic inspired by Deng et al. '
        'At each checkpoint the penultimate feature matrix is decomposed with SVD '
        '(fitted on training features only; test features are projected using the train-fitted basis). '
        'The <b>main component</b> retains the smallest number of leading singular directions '
        'whose cumulative singular-value mass reaches 80%; the <b>residual component</b> '
        'contains the remaining directions. '
        'Logistic regression probes are trained on the full, main, and residual feature sets. '
        'These are cheap frozen-feature probes; the neural network is not retrained.'
    ))

    p.append(_sub('Probe trajectories: full / main / residual'))
    sp_probe_b64 = _make_spectral_probes_fig_b64()
    sp_probe_img = _img_b64(spectral_pngs.get('probe_trajectories'))
    src = sp_probe_img or sp_probe_b64
    if src: p.append(f"<img src='{src}' style='max-width:100%;border:1px solid #ddd;border-radius:4px'>")
    else:   p.append('<p style=\'color:gray\'>Spectral probe trajectory not available.</p>')

    p.append(_note(
        'If the main component reaches near-final full-feature performance early, this is '
        '<i>consistent with</i> the interpretation that the dominant spectral component already '
        'carries much of the readout-accessible class information early in training. '
        'The residual component may contribute less to immediate linear readout performance '
        'but could continue to affect representational geometry and therefore CKA/NC/eNTK maturation. '
        'These observations are diagnostic evidence, not causal proof.'
    ))

    p.append(_sub('k_main over time'))
    img = _img_b64(spectral_pngs.get('k_main_over_time'))
    if img: p.append(f"<img src='{img}' style='max-height:280px;border:1px solid #ddd;border-radius:4px'>")

    p.append(_sub('Spectral mass of dominant directions'))
    img = _img_b64(spectral_pngs.get('spectral_mass_over_time'))
    if img: p.append(f"<img src='{img}' style='max-height:280px;border:1px solid #ddd;border-radius:4px'>")

    if spec_final_df is not None:
        p.append(_sub('Final-performance table by LR and component'))
        p.append(_html_table(spec_final_df, 'Spectral final performance (logistic probe)'))

    if 'spec_timing_df' in dir() and spec_timing_df is not None:
        p.append(_sub('Spectral operational timing (first epoch reaching 95% of final component probe acc)'))
        p.append(_html_table(spec_timing_df, 'Spectral timing events'))

    # Secondary: singular spectrum snapshots (collapsed)
    img = _img_b64(spectral_pngs.get('singular_spectrum_snapshots'))
    if img:
        p.append("<details style='margin:8px 0'><summary style='cursor:pointer;font-size:12px;color:#1565c0'>"
                 "Singular spectrum snapshots (secondary, click to expand)</summary>")
        p.append(f"<img src='{img}' style='max-width:100%;border:1px solid #ddd;border-radius:4px;margin-top:6px'>")
        p.append("</details>")

    # ── 8. Neural Collapse ────────────────────────────────────────────────────
    p.append(_section('8. NEURAL COLLAPSE'))
    b64 = _make_nc_fig_b64()
    if b64: p.append(f"<img src='{b64}' style='max-width:100%;border:1px solid #ddd;border-radius:4px'>")

    # ── 9. eNTK ───────────────────────────────────────────────────────────────
    p.append(_section('9. LAST-LAYER eNTK'))
    b64 = _make_entk_fig_b64()
    if b64: p.append(f"<img src='{b64}' style='max-width:100%;border:1px solid #ddd;border-radius:4px'>")

    # ── 10. Operational timing ────────────────────────────────────────────────
    p.append(_section('10. OPERATIONAL TIMING SUMMARIES'))
    p.append(_note(
        'Probe thresholds (95%/99%) are relative to each probe\'s own final-epoch accuracy, '
        'not absolute thresholds. CKA/eNTK thresholds are absolute. '
        'These epoch numbers describe when each metric operationally matures '
        'and do not constitute a definition of sufficiency.'
    ))
    try:
        pivot = timing_df.pivot(index='Event', columns='LR', values='Mean epoch').reset_index()
        p.append(_html_table(pivot, 'Mean epoch across seeds'))
    except Exception:
        p.append(_html_table(timing_df, 'Operational timing events'))

    # ── Footer ────────────────────────────────────────────────────────────────
    p.append(
        "<div style='font-size:10px;color:#90a4ae;margin-top:28px;"
        "border-top:1px solid #eee;padding-top:8px;font-family:monospace'>"
        "Multi-seed results dashboard — ResNet-18 / CIFAR-10 / 15 runs</div>"
    )

    full_html = '\n'.join(p)
    wrapper = (
        "<div style='max-height:90vh;overflow-y:auto;overflow-x:auto;"
        "padding:16px;border:1px solid #e0e0e0;border-radius:6px;"
        "background:#fff;font-family:sans-serif'>" + full_html + "</div>"
    )

    if save_html:
        out_path = GEN_DIR / 'dashboard.html'
        with open(out_path, 'w', encoding='utf-8') as fh:
            fh.write(f"<!DOCTYPE html><html><head><meta charset='utf-8'>"
                     f"<title>Multi-seed Results Dashboard</title></head>"
                     f"<body style='margin:0;padding:16px;background:#f8f8f8'>{full_html}</body></html>")
        print(f'Saved: {out_path}')

    display(HTML(wrapper))


build_dashboard(save_html=True)

In [ ]:
# ── Validation ────────────────────────────────────────────────────────────────
print('=== Validation ===')

REQUIRED_CROSS_LR = ['cross_lr_same_epoch','cross_lr_final','combined_final_sim','final_cka_matrix','within_lr_final']
REQUIRED_SPECTRAL  = ['spectral_summary','spectral_probes','spectral_sv']

all_ok = True
for k in REQUIRED_CROSS_LR:
    ok = agg.get(k) is not None
    print(f'  Cross-LR CKA [{"OK" if ok else "MISSING"}]  {k}')
    if not ok: all_ok = False

for k in REQUIRED_SPECTRAL:
    ok = agg.get(k) is not None
    print(f'  Spectral     [{"OK" if ok else "MISSING"}]  {k}')
    if not ok: all_ok = False

html_path = GEN_DIR / 'dashboard.html'
html_ok = html_path.exists()
print(f'  HTML         [{"OK" if html_ok else "MISSING"}]  {html_path}')
if not html_ok: all_ok = False

for k, v in spectral_pngs.items():
    ok = v is not None and v.exists()
    print(f'  Spectral PNG [{"OK" if ok else "MISSING"}]  {k}')

print()
print('=== Dashboard sections included ===')
for i, title in enumerate([
    'Run inventory',
    'Final performance across seeds',
    'Primary evidence: probes vs CKA',
    'Learning-rate comparison',
    'Within-LR seed robustness',
    'Cross-learning-rate CKA',
    'Spectral component analysis',
    'Neural Collapse',
    'Last-layer eNTK',
    'Operational timing summaries',
], start=1):
    print(f'  {i:2d}. {title}')

print()
print('=== Saved aggregate CSVs ===')
for p_csv in [
    AGG_DIR / 'spectral_final_performance_by_lr_component.csv',
    AGG_DIR / 'spectral_timing_events_by_lr_component.csv',
]:
    print(f'  [{"OK" if p_csv.exists() else "MISSING"}]  {p_csv.name}')

print()
print('All required files present.' if all_ok else 'WARNING: some required files are missing (see above).')